# PROJET OPEN DATA : Ingénierie des Données - Île-de-France (1/3)

**Objectif de ce Notebook :**
Extraire, nettoyer et fusionner les bases de données **DPE** et **ENEDIS** pour la région Île-de-France. 

**Auteurs :** Leopold Mopita  

In [1]:
import pandas as pd
import numpy as np
import re
from IPython.display import display, Markdown

# ==============================================================================
# 🌟 FONCTION D'AFFICHAGE ESTHÉTIQUE
# ==============================================================================
def afficher_bilan(titre, quoi, pourquoi, resultat, df_apercu=None):
    """Génère un affichage Markdown élégant pour résumer chaque étape dans le notebook."""
    texte = f"""
### {titre}
* 🛠️ **Ce qui a été fait :** {quoi}
* 💡 **Pourquoi :** {pourquoi}
* ✅ **Résultat :** {resultat}
    """
    display(Markdown(texte))
    
    if df_apercu is not None:
        display(Markdown("**🔎 Aperçu des données (5 premières lignes) :**"))
        # display() permet d'afficher les DataFrames sous forme de beaux tableaux HTML
        display(df_apercu.head()) 

afficher_bilan(
    titre="🚀 Initialisation réussie",
    quoi="Importation des librairies (Pandas, Numpy) et création du module d'affichage.",
    pourquoi="Pour manipuler les données et rendre les sorties du notebook esthétiques et lisibles.",
    resultat="L'environnement est prêt."
)


### 🚀 Initialisation réussie
* 🛠️ **Ce qui a été fait :** Importation des librairies (Pandas, Numpy) et création du module d'affichage.
* 💡 **Pourquoi :** Pour manipuler les données et rendre les sorties du notebook esthétiques et lisibles.
* ✅ **Résultat :** L'environnement est prêt.
    

##  Étape 1 : Le Nettoyeur d'Adresses
Pour fusionner nos bases, l'adresse est notre seule clé commune. Cependant, elle est souvent mal orthographiée ou formatée différemment selon les bases. Nous définissons ici une fonction stricte de standardisation.

In [2]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

DEPTS_IDF = ["75", "77", "78", "91", "92", "93", "94", "95"]
FICHIER_ENEDIS = "../enedis.csv"
FICHIER_DPE = "../dpe.csv"

In [3]:
# ==============================================================================
# FONCTION DE NETTOYAGE
# ==============================================================================
def clean_address_final(adresse):
    if pd.isna(adresse): return ""
    res = str(adresse).upper()
    res = re.sub(r'\b[0-9]{5}\b', '', res) # Supprime les codes postaux
    
    import unicodedata
    res =  unicodedata.normalize('NFD', res).encode('ascii', 'ignore').decode('utf-8') 

    res = re.sub(r'\bAVENUE\b', 'AV', res)
    res = re.sub(r'\bBOULEVARD\b', 'BD', res)
    res = re.sub(r'\bRUE\b', 'R', res)
    res = re.sub(r'\bCHEMIN\b', 'CH', res)
    res = re.sub(r'\bPLACE\b', 'PL', res)
    res = re.sub(r'\bQUAI\b', 'Q', res)
    res = re.sub(r'\bALLEE\b', 'AL', res)
    res = re.sub(r'\bSQUARE\b', 'SQ', res)
    res = re.sub(r'\bROUTE\b', 'RT', res)
    res = re.sub(r'\bVILLA\b', 'VL', res)
    res = re.sub(r'\bIMPASSE\b', 'IMP', res)
    res = re.sub(r'\bRESIDENCE\b', 'RES', res)
    res = re.sub(r'\bHAMEAU\b', 'HAM', res)
    res = re.sub(r'\bFERME\b', 'FER', res)
    res = re.sub(r'\bCITE\b', 'CITE', res)
    res = re.sub(r'\bPASSAGE\b', 'PASS', res)
    res = re.sub(r'\bPONT\b', 'PONT', res)
    res = re.sub(r'\b(DE|DU|DES|LE|LA|LES|L|AU|AUX|D)\b', '', res)
    res = re.sub(r'\s+', ' ', res).strip()
    res =  re.sub(r'[.,;-]', '', res)

    return res

afficher_bilan(
    titre="Étape 1 validée : Préparation du Nettoyeur",
    quoi="Définition de la liste des départements d'IDF et création d'une fonction Regex (`clean_address_final`).",
    pourquoi="Pour harmoniser les adresses (suppression des accents, majuscules, codes postaux) et garantir une jointure parfaite sans utiliser d'algorithmes flous (qui saturent la mémoire).",
    resultat="La fonction de nettoyage est prête à être appliquée aux millions de lignes."
)


### Étape 1 validée : Préparation du Nettoyeur
* 🛠️ **Ce qui a été fait :** Définition de la liste des départements d'IDF et création d'une fonction Regex (`clean_address_final`).
* 💡 **Pourquoi :** Pour harmoniser les adresses (suppression des accents, majuscules, codes postaux) et garantir une jointure parfaite sans utiliser d'algorithmes flous (qui saturent la mémoire).
* ✅ **Résultat :** La fonction de nettoyage est prête à être appliquée aux millions de lignes.
    

## Étape 2 : Extraction et Filtrage de la base DPE
Le fichier DPE contient des millions de lignes. Nous le lisons par morceaux pour éviter de saturer la mémoire vive (RAM), et nous ne conservons que les logements d'Île-de-France chauffés à l'électricité.

In [4]:
#variables intéressantes d'après les analyses de la region Nouvelle-Aquitaine

# 🛠️ MODIFICATION ICI : On ajoute les 4 colonnes de détail (chauffage, ecs, eclairage, refroidissement)
cols_dpe = [
    'etiquette_dpe', 'adresse_ban', 'code_postal_ban', 'conso_5 usages_ef', 
    'type_energie_principale_chauffage', 'surface_habitable_logement',
    'annee_construction', 'type_batiment',
    'conso_chauffage_ef', 'conso_ecs_ef', 'conso_eclairage_ef', 'conso_refroidissement_ef'
]

chunks = []
for chunk in pd.read_csv(FICHIER_DPE, sep=",", usecols=cols_dpe, chunksize=50000):
    chunk['dep'] = chunk['code_postal_ban'].astype(str).str.zfill(5).str[:2]
    chunk_idf = chunk[chunk['dep'].isin(DEPTS_IDF)].copy()
    
    if not chunk_idf.empty:
        chunk_elec = chunk_idf[chunk_idf['type_energie_principale_chauffage'].astype(str).str.contains('lectri', case=False, na=False)].copy()
        chunk_elec = chunk_elec[(chunk_elec['surface_habitable_logement'] > 9) & (chunk_elec['surface_habitable_logement'] < 300)]
        
        if not chunk_elec.empty:
            chunk_elec['adresse_join'] = chunk_elec['adresse_ban'].apply(clean_address_final)
            chunks.append(chunk_elec)

df_dpe = pd.concat(chunks) if chunks else pd.DataFrame()
if not df_dpe.empty:
    df_dpe = df_dpe.rename(columns={'conso_5 usages_ef': 'Conso_Theorique_kWh', 'surface_habitable_logement': 'Surface'})

afficher_bilan(
    titre="Étape 2 validée : Traitement du DPE",
    quoi="Lecture par blocs du fichier brut, filtrage géographique (IDF) et technique (Chauffage électrique, Surface 10-300m²). Ajout de l'année de construction et du type de bâtiment.",
    pourquoi="Pour ne comparer que ce qui est comparable (logements électriques) et éviter les valeurs aberrantes. La lecture par blocs évite le crash du PC.",
    resultat=f"{len(df_dpe):,} logements théoriques conservés.",
    df_apercu=df_dpe[['adresse_join', 'etiquette_dpe', 'Conso_Theorique_kWh', 'Surface', 'type_batiment']] if not df_dpe.empty else None
)


### Étape 2 validée : Traitement du DPE
* 🛠️ **Ce qui a été fait :** Lecture par blocs du fichier brut, filtrage géographique (IDF) et technique (Chauffage électrique, Surface 10-300m²). Ajout de l'année de construction et du type de bâtiment.
* 💡 **Pourquoi :** Pour ne comparer que ce qui est comparable (logements électriques) et éviter les valeurs aberrantes. La lecture par blocs évite le crash du PC.
* ✅ **Résultat :** 816,783 logements théoriques conservés.
    

**🔎 Aperçu des données (5 premières lignes) :**

,adresse_join,etiquette_dpe,Conso_Theorique_kWh,Surface,type_batiment
2,41 R HENRI BARBUSSE MONTFERMEIL,D,3639.9,32.2,appartement
13,7 RT GARDES MEUDON,D,4960.3,48.2,appartement
15,33B AV REPUBLIQUE MONTGERON,C,5184.6,70.8,appartement
16,43 AV JEAN JAURES COURNEUVE,E,4274.6,34.6,appartement
17,74 R BERCHERES PONTAULTCOMBAULT,C,4615.2,66.2,appartement


## Étape 3 : Extraction et Filtrage de la base ENEDIS
Nous traitons maintenant les données de consommation réelle. L'enjeu ici est de reconstruire l'adresse qui est éclatée sur plusieurs colonnes, puis de convertir l'énergie dans la bonne unité.

In [5]:
df_enedis = pd.read_csv(FICHIER_ENEDIS, sep=";", dtype=str, low_memory=False)

df_enedis['dep'] = df_enedis['Code Commune'].str.zfill(5).str[:2]
df_enedis = df_enedis[df_enedis['dep'].isin(DEPTS_IDF)].copy()

df_enedis['Année_num'] = pd.to_numeric(df_enedis['Année'], errors='coerce')
df_enedis = df_enedis[df_enedis['Année_num'] == df_enedis['Année_num'].max()].copy()

cols_addr = ['Numéro de voie', 'Indice de répétition', 'Type de voie', 'Libellé de voie', 'Nom Commune']
for c in cols_addr: 
    df_enedis[c] = df_enedis[c].fillna('').astype(str)

df_enedis['adresse_full'] = (df_enedis['Numéro de voie'] + " " + df_enedis['Indice de répétition'] + " " + 
                             df_enedis['Type de voie'] + " " + df_enedis['Libellé de voie'] + " " + df_enedis['Nom Commune'])

df_enedis['adresse_join'] = df_enedis['adresse_full'].apply(clean_address_final)

col_conso = "Consommation annuelle moyenne par logement de l'adresse (MWh)"
df_enedis[col_conso] = df_enedis[col_conso].astype(str).str.replace(',', '.').astype(float)
df_enedis['Conso_Reelle_kWh'] = df_enedis[col_conso] * 1000

afficher_bilan(
    titre="Étape 3 validée : Traitement d'ENEDIS",
    quoi="Filtrage sur l'IDF et sur l'année la plus récente. Concaténation des colonnes d'adresse, application du nettoyeur, et conversion des MWh en kWh.",
    pourquoi="Pour obtenir une adresse comparable à celle du DPE, et une unité d'énergie (kWh) identique pour permettre le calcul des écarts.",
    resultat=f"{len(df_enedis):,} immeubles franciliens conservés.",
    df_apercu=df_enedis[['adresse_join', 'Conso_Reelle_kWh', 'Nombre de logements']] if not df_enedis.empty else None
)


### Étape 3 validée : Traitement d'ENEDIS
* 🛠️ **Ce qui a été fait :** Filtrage sur l'IDF et sur l'année la plus récente. Concaténation des colonnes d'adresse, application du nettoyeur, et conversion des MWh en kWh.
* 💡 **Pourquoi :** Pour obtenir une adresse comparable à celle du DPE, et une unité d'énergie (kWh) identique pour permettre le calcul des écarts.
* ✅ **Résultat :** 159,825 immeubles franciliens conservés.
    

**🔎 Aperçu des données (5 premières lignes) :**

,adresse_join,Conso_Reelle_kWh,Nombre de logements
206870,4 R CHEVALIER BARRE ISSYMOULINEAUX,1630.0,21
206871,34 R DOCTEUR LOMBARD ISSYMOULINEAUX,2171.0,30
206872,30 R DOCTEUR LOMBARD ISSYMOULINEAUX,3373.0,14
206873,23 R DOCTEUR LOMBARD ISSYMOULINEAUX,5995.0,21
206874,2 ESPLANADE FONCET ISSYMOULINEAUX,1982.0,29


## Étape 4 : Jointure (Inner Join) et Exportation
Seuls les adresses parfaitement identiques entre la théorie (DPE) et la réalité (ENEDIS) seront conservées pour notre Data Mart final.

In [6]:
if not df_dpe.empty and not df_enedis.empty:
    # Agrégation des données du DPE par adresse (immeuble)
    dpe_grouped = df_dpe.groupby('adresse_join').agg({
        'Conso_Theorique_kWh': 'mean',
        'Surface': 'mean',
        'annee_construction': 'mean',
        'type_batiment': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
        'etiquette_dpe': lambda x: x.mode()[0] if not x.mode().empty else np.nan,
        'dep': 'first',
        'code_postal_ban': 'first',
        # 🛠️ AJOUT DES COLONNES ICI POUR NE PAS LES PERDRE LORS DU REGROUPEMENT :
        'conso_chauffage_ef': 'mean',
        'conso_ecs_ef': 'mean',
        'conso_eclairage_ef': 'mean',
        'conso_refroidissement_ef': 'mean'
    }).reset_index()

    # Jointure
    df_idf = pd.merge(df_enedis, dpe_grouped, on='adresse_join', how='inner')
    df_idf = df_idf[df_idf['etiquette_dpe'].isin(['A','B','C','D','E','F','G'])]
    
    # Export du fichier propre
    NOM_FICHIER_PROPRE = "Data_IDF_Propre.csv"
    df_idf.to_csv(NOM_FICHIER_PROPRE, index=False)
    
    afficher_bilan(
        titre="🎉 Étape 4 validée : Le Data Mart est prêt !",
        quoi="Jointure interne (`inner_join`) entre ENEDIS et le DPE sur la clé `adresse_join`. Sauvegarde du résultat en CSV.",
        pourquoi="Pour créer la base de données finalisée qui servira à l'Analyse et à la Modélisation.",
        resultat=f"{len(df_idf):,} immeubles croisés et sauvegardés dans '{NOM_FICHIER_PROPRE}'.",
        df_apercu=df_idf[['adresse_join', 'Conso_Reelle_kWh', 'Conso_Theorique_kWh', 'etiquette_dpe', 'type_batiment']]
    )
else:
    print("❌ Erreur : L'une des bases est vide.")


### 🎉 Étape 4 validée : Le Data Mart est prêt !
* 🛠️ **Ce qui a été fait :** Jointure interne (`inner_join`) entre ENEDIS et le DPE sur la clé `adresse_join`. Sauvegarde du résultat en CSV.
* 💡 **Pourquoi :** Pour créer la base de données finalisée qui servira à l'Analyse et à la Modélisation.
* ✅ **Résultat :** 54,302 immeubles croisés et sauvegardés dans 'Data_IDF_Propre.csv'.
    

**🔎 Aperçu des données (5 premières lignes) :**

,adresse_join,Conso_Reelle_kWh,Conso_Theorique_kWh,etiquette_dpe,type_batiment
0,30 R DOCTEUR LOMBARD ISSYMOULINEAUX,3373.0,3081.513333,D,appartement
1,23 R DOCTEUR LOMBARD ISSYMOULINEAUX,5995.0,4045.752381,C,appartement
2,64 AV GENERAL GAULLE ISSYMOULINEAUX,5206.0,5368.912500,D,appartement
3,67 PROMENADE VERGER ISSYMOULINEAUX,1809.0,2061.216667,B,appartement
4,40 PROMENADE VERGER ISSYMOULINEAUX,1837.0,2804.816667,B,appartement
